# parity

The acceptance test, and the last step of `run_pipeline`: the two engines built the same
models from the same files, so their gold tables must agree. Each `run` left a fingerprint
of `fct_summary` in `dbt_landing/Files/parity/<run_id>/`; this compares the two and fails
the run if they differ.

- **Exact** fields must be identical: row count, DUIDs, days, first and last date.
- **Money** sums get a relative tolerance of 1e-7, because two engines can compute the same
  thing and still disagree in the last place: `DOUBLE -> DECIMAL` rounds HALF_UP on Spark and
  otherwise in T-SQL, and summing the same doubles in another order moves the last bit.

The tolerance absorbs rounding. It is not there to paper over a logic difference: a leg
that drifts past it is a finding, not a threshold to raise.

In [ ]:
# Set by run_pipeline: the run whose fingerprints are compared.
run_id = "manual"

In [ ]:
# 1. SETTINGS
import json
from pathlib import Path

import notebookutils

ENGINES = ("dwh", "spark")
EXACT = ["rows_total", "duids", "days", "date_min", "date_max"]
APPROX = ["mw_sum", "price_sum"]
REL_TOL = 1e-7

vl = notebookutils.variableLibrary.getLibrary("deploy_config")
if vl.engines != "all":
    notebookutils.notebook.exit(f"parity skipped: only {vl.engines} was built")

In [ ]:
# 2. THIS RUN'S FINGERPRINTS, by run id. Files/parity also holds every earlier run's, and a
# leg that failed this run must not be graded on a stale one.
folder = Path(f"/lakehouse/default/Files/parity/{run_id}")
fingerprints = {}
for engine in ENGINES:
    path = folder / f"{engine}.json"
    if not path.exists():
        raise RuntimeError(f"no fingerprint for {engine} in run {run_id}")
    fingerprints[engine] = json.loads(path.read_text())

header = "field".ljust(14) + "".join(e.rjust(20) for e in ENGINES)
print(header)
print("-" * len(header))
for field in EXACT + APPROX:
    row = field.ljust(14)
    for e in ENGINES:
        v = fingerprints[e].get(field)
        row += (f"{v:,}" if isinstance(v, int) else f"{v:.6f}" if isinstance(v, float) else str(v)).rjust(20)
    print(row)

In [ ]:
# 3. COMPARE
base_name, other_name = ENGINES
base, other = fingerprints[base_name], fingerprints[other_name]
failures = []
for field in EXACT:
    if other.get(field) != base.get(field):
        failures.append(f"{field}: {base_name}={base.get(field)!r} but {other_name}={other.get(field)!r}")
for field in APPROX:
    a, b = float(base.get(field, 0)), float(other.get(field, 0))
    scale = max(abs(a), abs(b), 1.0)
    if abs(a - b) / scale > REL_TOL:
        failures.append(f"{field}: {base_name}={a!r} but {other_name}={b!r} "
                        f"(relative difference {abs(a - b) / scale:.3e} > {REL_TOL:.0e})")

if failures:
    # Before changing a tolerance, check the two usual causes:
    #   * a string join key with a trailing space. T-SQL pads on comparison, Spark does
    #     not, and that changes row counts, not just sums.
    #   * one engine's model drifting from the shared logic. Diff models/aemo/dwh/ against
    #     models/aemo/spark/.
    raise RuntimeError("PARITY FAILED, the engines did not produce the same gold layer:\n  "
                       + "\n  ".join(failures))

print(f"\nPARITY OK: {base_name} and {other_name} agree on {base['model']} "
      f"({base['rows_total']:,} rows, {base['days']} days)")